In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
def clean_generic_name(name):
    if pd.isna(name) or name == 'nan':
        return name
    
    # 1. 統一標點符號：全型斜線與逗號轉半型，移除常見括號內容
    name = name.replace('／', '/').replace('，', ',')
    name = re.sub(r'[\(\uff08].*?[\)\uff09]', '', name)
    
    # 2. 強力移除劑量與單位 (處理 100,000, 1%, 80 mg/ml, 2.4 MIU 等)
    # 匹配數字、逗號、點、百分比，後接常見容量單位
    name = re.sub(r'[\d\.,\s]+(g|mg|ml|units|miu|％|%)\b', ' ', name, flags=re.IGNORECASE)
    # 針對單獨殘留的 /ml 或 %
    name = re.sub(r'/\s*ml|%', '', name, flags=re.IGNORECASE)
    
    # 3. 移除特定劑型關鍵字與無義詞
    forms = ['inj', 'tab', 'cap', 'susp', 'iv', 'oral', 'for', 'solution', 'hydrate', 'sodium', 'benzathine']
    pattern_forms = r'\b(' + '|'.join(forms) + r')\b'
    name = re.sub(pattern_forms, '', name, flags=re.IGNORECASE)

    # 4. 特定藥名結構處理 (依據您的要求)
    # Amphotericin B liposome -> Amphotericin B/liposome
    name = re.sub(r'Amphotericin B liposome', 'Amphotericin B/liposome', name, flags=re.IGNORECASE)

    # 5. 【核心強化】消除斜線 (/) 前後的任何空格
    # \s* 代表 0 到多個空白字元
    name = re.sub(r'\s*/\s*', '/', name)

    
    # 5. 清理殘留符號：移除多餘空格、末尾點號與斜線
    name = re.sub(r'\s+', ' ', name) # 多空格轉單空格
    name = name.strip(' ./,')       # 移除前後的空格、點、斜線、逗號
    
    # 6. 字典對照 (處理特殊轉換)
    mapping = {
        'R +I': 'ifampin/Isoniazid',
        'R 300 +I 150': 'ifampin/Isoniazid',
        'Penicillin G .': 'Penicillin G',
        'Penicillin G benzathine': 'Penicillin G',
        'Baktar': 'Sulfamethoxazole/Trimethoprim',
        'Clindamycin 1 ％' : 'Clindamycin',
        'Penicillin 5 MU' : 'Penicillin',
        'Rifampin, Isoniazid and Ethambutol' : 'Rifampin/Isoniazid/Ethambutol',
        'Minocycline injection' : 'Minocycline',
        'Amoxicillin/Clavulanate': 'Amoxicillin/Clavulanic acid'
    }
    
    # 如果完全符合字典 key，或是處理後變成 key 的樣子就轉換
    return mapping.get(name, name)

In [3]:
df2024 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2024\2024Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})
df2025 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2025\2025Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})

# df2024['VERIFYDATE'] = pd.to_datetime(df2024['VERIFYDATE'], format='%Y%m%d', errors='coerce')
df2025['VERIFYDATE'] = pd.to_datetime(df2025['VERIFYDATE'], format='%Y%m%d', errors='coerce')
# df2024['STARTTIME'] = pd.to_datetime(df2024['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
# df2024['ENDTIME'] = pd.to_datetime(df2024['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['STARTTIME'] = pd.to_datetime(df2025['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['ENDTIME'] = pd.to_datetime(df2025['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')

# table14 = pd.concat([df2024, df2025], ignore_index=True)

table14 = df2025

In [4]:
table14 = table14.dropna(how='all')
table14 = table14[table14['ACCOUNTNO'].notna()]
table14 = table14.loc[:, ~table14.columns.str.contains('^Unnamed')]

In [5]:
len(table14), len(table14['ACCOUNTNO'].unique())

(37073, 12745)

In [6]:
table14['VERIFYDATE'] = pd.to_datetime(table14['VERIFYDATE'], format='%Y%m%d%H%M', errors='coerce')
table14['STARTTIME'] = pd.to_datetime(table14['STARTTIME'], format='%Y%m%d%H%M', errors='coerce')
table14['ENDTIME'] = pd.to_datetime(table14['ENDTIME'], format='%Y%m%d%H%M', errors='coerce')

In [7]:
# table14[table14['ACCOUNTNO'] == 'I11300000002']

In [8]:
table14['GENERICNAME_Clear'] = table14['GENERICNAME'].apply(clean_generic_name)

In [9]:
table14['GENERICNAME_Clear'].unique()

array(['Cephalexin', 'Fenticonazole', 'Flomoxef', 'Cefepime',
       'Ceftazidime', 'Ciprofloxacin', 'Amoxicillin/Clavulanic acid',
       'Peramivir', 'Metronidazole', 'Cefixime', 'Ceftriaxone',
       'Oseltamivir', 'Baloxavir marboxil', 'Azithromycin', 'Cefazolin',
       'Piperacillin/Tazobactam', 'Cefadroxil', 'Vancomycin',
       'Levofloxacin', 'Amoxicillin', 'Clindamycin', 'Nemonoxacin',
       'Cefoperazone/sulbactam', 'Ampicillin', 'Cefuroxime', 'Gentamicin',
       'Doxycycline', 'Sulfamethoxazole/Trimethoprim', 'Acyclovir',
       'Moxifloxacin', 'Teicoplanin', 'Pipemidic acid', 'Linezolid',
       'Fosfomycin', 'tenofovir/emtricitabine/bictegravir',
       'Ampicillin/Sulbactam', 'Telbivudine', 'Tenofovir alafenamide',
       'Penicillin', 'Cefotaxime', 'Pyrazinamide',
       'Rifampin/Isoniazid/Ethambutol', 'Rifampin', 'Ethambutol',
       'Isoniazid', 'Fluconazole', 'Nystatin', 'Micafungin',
       'Valaciclovir', 'Terbinafine', 'Erythromycin', 'Ertapenem',
       'Minoc

In [10]:
table14['GENERICNAME_Clear'][table14['GENERICNAME_Clear'].isna() == True]

Series([], Name: GENERICNAME_Clear, dtype: object)

In [11]:
table14[['ACCOUNTNO', 'GENERICNAME_Clear']]

,ACCOUNTNO,GENERICNAME_Clear
0,I11400000003,Cephalexin
1,I11400000003,Fenticonazole
2,I11400000004,Flomoxef
3,I11400000004,Cefepime
4,I11400000004,Cefepime
...,...,...
37068,I11400060720,Piperacillin/Tazobactam
37069,I11400060731,Flomoxef
37070,I11400060731,Flomoxef
37071,I11400060731,Flomoxef


In [12]:
infection_cols = [
    "INFECTIONSITE1",
    "INFECTIONSITE2",
    "INFECTIONSITE3",
    "INFECTIONSITE4",
    "INFECTIONSITE5",
    "INFECTIONSITE9"
]

for col in infection_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [13]:
table14['INFECTIONSITE1']

0        0
1        0
2        0
3        0
4        0
        ..
37068    1
37069    1
37070    0
37071    1
37072    1
Name: INFECTIONSITE1, Length: 37073, dtype: int32

In [14]:
BACTERIA_cols = [
    "BACTERIA1",
    "BACTERIA2",
    "BACTERIA3",
    "BACTERIA4",
    "BACTERIA5",
    "BACTERIA9"
]

for col in BACTERIA_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [15]:
table14['BACTERIA1']

0        0
1        0
2        1
3        1
4        0
        ..
37068    0
37069    0
37070    0
37071    0
37072    0
Name: BACTERIA1, Length: 37073, dtype: int32

In [16]:
table14['AUTIBIOTICRANK'].unique()

array(['A11', 'A32', 'A42', 'A21', 'A22', 'A52', 'C42'], dtype=object)

In [17]:
rank_mapping = {
    'A11': 1,
    'A21': 2,
    'A22': 2,
    'A32': 2,
    'A42': 3,
    'A52': 3,
    'C42': 3
}

In [18]:
table14['AUTIBIOTIC_GROUP'] = table14['AUTIBIOTICRANK'].map(rank_mapping)

In [19]:
last_index = table14.groupby('ACCOUNTNO')['VERIFYDATE'].min().reset_index()
table14_last = pd.merge(table14, last_index, on=['ACCOUNTNO', 'VERIFYDATE'])

In [20]:
# first_time = table14.groupby('ACCOUNTNO')['VERIFYDATE'].transform('min')

# # 0~1 day
# table14_D1 = table14[(table14['VERIFYDATE'] >= first_time) & 
#              (table14['VERIFYDATE'] < first_time + pd.Timedelta(days=1))]
# # 1~2 day
# table14_D2 = table14[(table14['VERIFYDATE'] >= first_time + pd.Timedelta(days=1)) & 
#              (table14['VERIFYDATE'] < first_time + pd.Timedelta(days=2))]

In [21]:
# d1 = (table14_D1.groupby('ACCOUNTNO')['GENERICNAME_Clear'].agg(lambda x: set(x)))
# d2 = (table14_D2.groupby('ACCOUNTNO')['GENERICNAME_Clear'].agg(lambda x: set(x)))

# abx_compare = pd.concat([d1, d2], axis=1)
# abx_compare.columns = ['Day1', 'Day2']

In [22]:
# abx_compare['Day1'] = abx_compare['Day1'].apply(lambda x: x if isinstance(x, set) else set())
# abx_compare['Day2'] = abx_compare['Day2'].apply(lambda x: x if isinstance(x, set) else set())

# # 集合+法
# abx_compare['add'] = abx_compare.apply(lambda d: d['Day2'] - d['Day1'], axis=1)

# abx_compare['remove'] = abx_compare.apply(lambda d: d['Day1'] - d['Day2'], axis=1)

In [23]:
# abx_compare.to_csv('abx_compare.csv')

In [24]:
# # 計算 Day1 被保留的比例
# abx_compare2 = abx_compare[abx_compare['Day2'].notna() & (abx_compare['Day2'] != '')].copy()

# # retention
# # abx_compare2['preserved rate'] = abx_compare2.apply(lambda r: len(r['Day1'] & r['Day2']) / len(r['Day1']) 
# #                                       if len(r['Day1']) > 0 else np.nan, axis=1)

# # Jaccard
# abx_compare2['preserved rate'] = abx_compare2.apply(lambda r: len(r['Day1'] & r['Day2']) / len(r['Day1'] | r['Day2']) 
#                                       if len(r['Day1']) > 0 else np.nan, axis=1)

In [25]:
# abx_compare2.to_csv('Day1 Day2 preserved rate.csv')

In [26]:
# abx_compare2['preserved rate'].mean()

In [27]:
# abx_exploded = abx_compare.explode('Day1')

# abx_exploded['is retain'] = abx_exploded.apply(lambda x: x['Day1'] in x['Day2'] 
#                                              if isinstance(x['Day2'], (set, list)) else x['Day1'] == x['Day2'], axis=1)

# summary = abx_exploded.groupby('Day1').agg(D1病患數=('Day1','count'),D2仍存在=('is retain','sum')).reset_index()
# summary.columns = ['Drug', 'D1病患數', 'D2仍存在']
# summary['retention'] = summary['D2仍存在'] / summary['D1病患數']
# summary = summary.sort_values(by='D1病患數', ascending=False)

In [28]:
# summary.head(10)

In [29]:
# abx_compare.to_csv('abx_compare.csv')

In [30]:
# len(table14_last['GENERICNAME_Clear'].unique())

In [31]:
table14_last = table14_last[['ACCOUNTNO', 'GENERICNAME_Clear']].drop_duplicates()

In [32]:
table14_last.head(10)

,ACCOUNTNO,GENERICNAME_Clear
0,I11400000003,Cephalexin
1,I11400000003,Fenticonazole
2,I11400000004,Flomoxef
3,I11400000004,Cefepime
5,I11400000008,Ciprofloxacin
6,I11400000013,Amoxicillin/Clavulanic acid
7,I11400000019,Peramivir
9,I11400000026,Amoxicillin/Clavulanic acid
10,I11400000034,Metronidazole
11,I11400000034,Cefixime


In [33]:
table14_last['GENERICNAME_Clear'].unique(), len(table14_last['GENERICNAME_Clear'].unique())

(array(['Cephalexin', 'Fenticonazole', 'Flomoxef', 'Cefepime',
        'Ciprofloxacin', 'Amoxicillin/Clavulanic acid', 'Peramivir',
        'Metronidazole', 'Cefixime', 'Ceftriaxone', 'Oseltamivir',
        'Baloxavir marboxil', 'Azithromycin', 'Cefazolin', 'Cefadroxil',
        'Amoxicillin', 'Clindamycin', 'Nemonoxacin',
        'Cefoperazone/sulbactam', 'Ampicillin', 'Piperacillin/Tazobactam',
        'Gentamicin', 'Sulfamethoxazole/Trimethoprim', 'Acyclovir',
        'Levofloxacin', 'Cefuroxime', 'Doxycycline', 'Pipemidic acid',
        'Moxifloxacin', 'tenofovir/emtricitabine/bictegravir',
        'Ampicillin/Sulbactam', 'Pyrazinamide',
        'Rifampin/Isoniazid/Ethambutol', 'Penicillin', 'Fosfomycin',
        'Ceftazidime', 'Valaciclovir', 'Isoniazid', 'Vancomycin',
        'Tenofovir alafenamide', 'Cefotaxime', 'Fluconazole', 'Ertapenem',
        'Famciclovir', 'Ceftizoxime', 'Erythromycin', 'Meropenem',
        'Linezolid', 'Nystatin', 'Clarithromycin',
        'tenofovir/emt

In [34]:
abx14 = (table14_last.assign(value=1)
                     .pivot_table(index=['ACCOUNTNO'],columns='GENERICNAME_Clear',values='value',fill_value=0))

In [35]:
# add = table14_last.groupby('ACCOUNTNO')['AUTIBIOTIC_GROUP'].max().reset_index()
# abx14 = abx14.merge(add, on='ACCOUNTNO', how='left')

In [36]:
# infection_cols = ['INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3', 
#                   'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9']

# for col in infection_cols:
#     table14[col] = table14[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

# binary OTHERINFECTIONSITE 
table14['OTHERINFECTIONSITE_flag'] = (
    table14['OTHERINFECTIONSITE'].fillna('').str.strip().ne('').astype(int)
)

infects_summary = table14.groupby('ACCOUNTNO').agg({
    **{col: 'max' for col in infection_cols}, 
    'OTHERINFECTIONSITE_flag': 'max'
}).reset_index()


abx14 = abx14.merge(infects_summary, on='ACCOUNTNO', how='left').fillna(0)

In [37]:
abx14.columns

Index(['ACCOUNTNO', 'Acyclovir', 'Amikacin', 'Amoxicillin',
       'Amoxicillin/Clavulanic acid', 'Ampicillin', 'Ampicillin/Sulbactam',
       'Azithromycin', 'Baloxavir marboxil', 'Cefadroxil', 'Cefazolin',
       'Cefepime', 'Cefixime', 'Cefoperazone/sulbactam', 'Cefotaxime',
       'Cefoxitin', 'Ceftazidime', 'Ceftazidime/Avibactam', 'Ceftizoxime',
       'Ceftriaxone', 'Cefuroxime', 'Cephalexin', 'Ciprofloxacin',
       'Clarithromycin', 'Clindamycin', 'Dicloxacillin', 'Doripenem',
       'Doxycycline', 'Ertapenem', 'Erythromycin', 'Famciclovir',
       'Fenticonazole', 'Flomoxef', 'Fluconazole', 'Fosfomycin', 'Ganciclovir',
       'Gentamicin', 'Imipenem/Cilastatin', 'Isoniazid', 'Itraconazole',
       'Levofloxacin', 'Linezolid', 'Meropenem', 'Metronidazole',
       'Minocycline', 'Moxifloxacin', 'Nemonoxacin', 'Nystatin', 'Oseltamivir',
       'Oxacillin', 'Penicillin', 'Peramivir', 'Pipemidic acid',
       'Piperacillin/Tazobactam', 'Pyrazinamide',
       'Rifampin/Isoniazid/Et

In [38]:
abx14

,ACCOUNTNO,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Ampicillin,Ampicillin/Sulbactam,Azithromycin,Baloxavir marboxil,Cefadroxil,...,Vancomycin,tenofovir/emtricitabine,tenofovir/emtricitabine/bictegravir,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag
0,I11400000003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
1,I11400000004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
2,I11400000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
3,I11400000013,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
4,I11400000019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12740,I11400060686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
12741,I11400060687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
12742,I11400060701,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
12743,I11400060720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0


In [39]:
table14_clear = abx14.reset_index(drop=True)

In [40]:
# table14_clear.to_csv('table14_clear.csv', index=False)

In [41]:
table14_clear = table14_clear.drop(columns=['ACCOUNTNO']) # 'AUTIBIOTIC_GROUP'

In [43]:
start_index = table14_clear.columns.get_loc('Acyclovir')
end_index = table14_clear.columns.get_loc('tenofovir/emtricitabine/bictegravir') # 2025: tenofovir/emtricitabine/bictegravir

abx_cols = table14_clear.columns[start_index:end_index+1]
col_sum = table14_clear[abx_cols].sum()

final_cols = col_sum[col_sum >= 0].index.tolist() # 抗生素
base_cols = [c for c in table14_clear.columns if c not in abx_cols] # 感染部位
data_filter = table14_clear[base_cols + final_cols]

feature_cols = list(set(data_filter.columns) - set(abx_cols))
X = data_filter[feature_cols]
y = data_filter[final_cols]

X.shape, X.sum(), y.shape, y.sum()

((12745, 7),
 INFECTIONSITE4              103
 INFECTIONSITE3             1397
 OTHERINFECTIONSITE_flag     256
 INFECTIONSITE9              303
 INFECTIONSITE5              664
 INFECTIONSITE2             2070
 INFECTIONSITE1             2475
 dtype: int64,
 (12745, 63),
 Acyclovir                                74.0
 Amikacin                                  3.0
 Amoxicillin                             185.0
 Amoxicillin/Clavulanic acid            2471.0
 Ampicillin                              112.0
                                         ...  
 Terbinafine                               1.0
 Valaciclovir                             60.0
 Vancomycin                               24.0
 tenofovir/emtricitabine                   3.0
 tenofovir/emtricitabine/bictegravir      41.0
 Length: 63, dtype: float64)

In [44]:
abx14 = y

result = {}
for i in abx14.columns:
    summ = abx14[i].sum()
    if summ >= 0:
       result[i] = summ

for key, value in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f'{key}: {value}')
    
print(len(result))

Flomoxef: 2480.0
Amoxicillin/Clavulanic acid: 2471.0
Cefixime: 1091.0
Ciprofloxacin: 1049.0
Cefazolin: 908.0
Cefuroxime: 845.0
Azithromycin: 746.0
Piperacillin/Tazobactam: 722.0
Baloxavir marboxil: 613.0
Cefoperazone/sulbactam: 589.0
Cefadroxil: 514.0
Peramivir: 510.0
Metronidazole: 474.0
Clindamycin: 328.0
Oseltamivir: 301.0
Levofloxacin: 257.0
Cephalexin: 249.0
Ceftriaxone: 226.0
Gentamicin: 222.0
Amoxicillin: 185.0
Doxycycline: 143.0
Ampicillin: 112.0
Nemonoxacin: 92.0
Acyclovir: 74.0
Valaciclovir: 60.0
Tenofovir alafenamide: 55.0
Cefotaxime: 42.0
tenofovir/emtricitabine/bictegravir: 41.0
Cefepime: 40.0
Ceftazidime: 32.0
Fosfomycin: 30.0
Vancomycin: 24.0
Sulfamethoxazole/Trimethoprim: 23.0
Famciclovir: 22.0
Moxifloxacin: 15.0
Ampicillin/Sulbactam: 14.0
Pipemidic acid: 14.0
Nystatin: 12.0
Clarithromycin: 10.0
Fluconazole: 9.0
Meropenem: 8.0
Fenticonazole: 7.0
Penicillin: 7.0
Dicloxacillin: 6.0
Minocycline: 6.0
Cefoxitin: 5.0
Ceftizoxime: 5.0
Oxacillin: 5.0
Amikacin: 3.0
Erythromycin:

In [45]:
col_sum = abx14.sum()

abx14_filter = abx14.loc[:, col_sum >= 0]

In [46]:
abx14_filter.columns

Index(['Acyclovir', 'Amikacin', 'Amoxicillin', 'Amoxicillin/Clavulanic acid',
       'Ampicillin', 'Ampicillin/Sulbactam', 'Azithromycin',
       'Baloxavir marboxil', 'Cefadroxil', 'Cefazolin', 'Cefepime', 'Cefixime',
       'Cefoperazone/sulbactam', 'Cefotaxime', 'Cefoxitin', 'Ceftazidime',
       'Ceftazidime/Avibactam', 'Ceftizoxime', 'Ceftriaxone', 'Cefuroxime',
       'Cephalexin', 'Ciprofloxacin', 'Clarithromycin', 'Clindamycin',
       'Dicloxacillin', 'Doripenem', 'Doxycycline', 'Ertapenem',
       'Erythromycin', 'Famciclovir', 'Fenticonazole', 'Flomoxef',
       'Fluconazole', 'Fosfomycin', 'Ganciclovir', 'Gentamicin',
       'Imipenem/Cilastatin', 'Isoniazid', 'Itraconazole', 'Levofloxacin',
       'Linezolid', 'Meropenem', 'Metronidazole', 'Minocycline',
       'Moxifloxacin', 'Nemonoxacin', 'Nystatin', 'Oseltamivir', 'Oxacillin',
       'Penicillin', 'Peramivir', 'Pipemidic acid', 'Piperacillin/Tazobactam',
       'Pyrazinamide', 'Rifampin/Isoniazid/Ethambutol',
       'S

In [47]:
mask = abx14_filter.sum(axis=1) > 0
abx14_final = abx14_filter[mask].reset_index()

In [48]:
abx14_final

,index,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Ampicillin,Ampicillin/Sulbactam,Azithromycin,Baloxavir marboxil,Cefadroxil,...,Pyrazinamide,Rifampin/Isoniazid/Ethambutol,Sulfamethoxazole/Trimethoprim,Telbivudine,Tenofovir alafenamide,Terbinafine,Valaciclovir,Vancomycin,tenofovir/emtricitabine,tenofovir/emtricitabine/bictegravir
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12740,12740,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12741,12741,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12742,12742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12743,12743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
